In [ ]:
import pandas as pd
import glob
import os
import numpy as np
import matplotlib.pyplot as plt

# =========================
# CONFIG (Convolutional MNIST)
# =========================
BASE_DIR_ROOT = r"C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-MNIST"

PRUNE_LAYERS_OPTIONS = ["CONV", "FHL", "SHL", "FHL+SHL", "ALL"]

PRUNE_LAYER_DIR_MAP = {
    "CONV":    "prune_layers_CONV",
    "FHL":     "prune_layers_FHL",
    "SHL":     "prune_layers_SHL",
    "FHL+SHL": "prune_layers_FHL+SHL",
    "ALL":     "prune_layers_ALL",
}

BATCH_DIR_TEMPLATE = "p-percentage_{}\\batch_size_{}"
FILE_PATTERN = "convol_{}_{}_run_*"

BATCH_SIZES = [64, 1024]
PRUNING_PERCENTAGES = [
    0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7,
    0.8, 0.82, 0.84, 0.86, 0.88, 0.9, 0.92, 0.94, 0.96, 0.98, 0.985, 0.99, 1.0
]

LN10 = np.log(10)

# =========================
# STEP 1 — AVERAGE RAW RUNS → SAVE CSV
# For each (prune_layer, p, bs): read all raw .txt files,
# average by Batch_Number, save as averaged_runs_conv_{prune_layer}_p_{p}_bs_{bs}.csv
# Skips if file already exists (re-run safe).
# =========================
for prune_layer in PRUNE_LAYERS_OPTIONS:
    base_dir = os.path.join(BASE_DIR_ROOT, PRUNE_LAYER_DIR_MAP[prune_layer])

    for bs in BATCH_SIZES:
        for p in PRUNING_PERCENTAGES:
            folder = os.path.join(base_dir, BATCH_DIR_TEMPLATE.format(p, bs))
            out_csv = os.path.join(folder, f"averaged_runs_conv_{prune_layer}_p_{p}_bs_{bs}.csv")

            if os.path.isfile(out_csv):
                print(f"[SKIP] already exists: {os.path.basename(out_csv)}")
                continue

            files = glob.glob(os.path.join(folder, FILE_PATTERN.format(p, bs)))
            if not files:
                print(f"[WARNING] No raw files for {prune_layer}, p={p}, bs={bs}")
                continue

            dfs = []
            for fpath in files:
                df = pd.read_csv(fpath, sep=r"\s+")
                df.columns = df.columns.str.strip()
                df["CE_Train"]    = pd.to_numeric(df["CE_Train"],    errors="coerce")
                df["CE_TEST"]     = pd.to_numeric(df["CE_TEST"],     errors="coerce")
                df["Accuracy(%)"] = pd.to_numeric(df["Accuracy(%)"], errors="coerce")
                dfs.append(df)

            all_runs = pd.concat(dfs, ignore_index=True)
            avg_df = all_runs.groupby("Batch_Number", as_index=False).agg(
                Avg_CE_Train=("CE_Train",    "mean"),
                Avg_CE_Test= ("CE_TEST",     "mean"),
                Avg_Accuracy=("Accuracy(%)", "mean"),
                Num_Runs=    ("CE_TEST",     "count"),
            )
            avg_df.to_csv(out_csv, index=False)
            print(f"[SAVED] {out_csv}")


In [ ]:
# =========================
# STEP 2 — PLOT averaged CE curves (CE_Test) per prune_layer × batch_size
# =========================
PRUNING_COLOR_MAP = {
    0.0: "#17becf", 0.1: "#B9D9EB", 0.2: "#bcbd22", 0.3: "#7f7f7f",
    0.4: "#e377c2", 0.5: "#8c564b", 0.6: "#800080", 0.7: "#d62728",
    0.8: "#2ca02c", 0.82: "#66c2a5", 0.84: "#fc8d62", 0.86: "#8da0cb",
    0.88: "#e78ac3", 0.9: "#C5B0D5", 0.92: "#8DD3C7", 0.94: "#BEBADA",
    0.96: "#80B1D3", 0.98: "#FDB462", 0.985: "#6A5ACD", 0.99: "#FF6F61",
    1.0: "#1f77b4",
}

plt.rcParams.update({
    "font.size": 18, "axes.titlesize": 18, "axes.labelsize": 18,
    "xtick.labelsize": 18, "ytick.labelsize": 18, "legend.fontsize": 13,
})

PLOT_ROOT = os.path.join(BASE_DIR_ROOT, "Avg_Plots")
os.makedirs(PLOT_ROOT, exist_ok=True)

for prune_layer in PRUNE_LAYERS_OPTIONS:
    base_dir = os.path.join(BASE_DIR_ROOT, PRUNE_LAYER_DIR_MAP[prune_layer])
    plot_dir = os.path.join(PLOT_ROOT, prune_layer)
    os.makedirs(plot_dir, exist_ok=True)

    for bs in BATCH_SIZES:
        avg_dfs = {}
        for p in PRUNING_PERCENTAGES:
            folder = os.path.join(base_dir, BATCH_DIR_TEMPLATE.format(p, bs))
            fpath  = os.path.join(folder, f"averaged_runs_conv_{prune_layer}_p_{p}_bs_{bs}.csv")
            if os.path.isfile(fpath):
                avg_dfs[p] = pd.read_csv(fpath)

        if not avg_dfs:
            continue

        # CE Test plot
        plt.figure(figsize=(12, 6))
        for p, df in sorted(avg_dfs.items()):
            color = PRUNING_COLOR_MAP.get(p, "#999999")
            plt.plot(df["Batch_Number"], df["Avg_CE_Test"],
                     label=f"P%={int(p*100)}" if p*100 == int(p*100) else f"P%={p*100:.1f}",
                     color=color)
        plt.axhline(y=LN10, color="black", linestyle="--", linewidth=1)
        plt.xlabel("Batch Number")
        plt.ylabel("Average CE Test")
        plt.title(f"Conv-MNIST — CE Test\nPrune: {prune_layer}, BS={bs}")
        plt.grid(True)
        plt.tight_layout()
        out_png = os.path.join(plot_dir, f"CE_Test_Avg_{prune_layer}_BS{bs}.png")
        plt.savefig(out_png, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"[PLOT] {out_png}")

        # CE Train plot
        plt.figure(figsize=(12, 6))
        for p, df in sorted(avg_dfs.items()):
            color = PRUNING_COLOR_MAP.get(p, "#999999")
            plt.plot(df["Batch_Number"], df["Avg_CE_Train"],
                     label=f"P%={int(p*100)}" if p*100 == int(p*100) else f"P%={p*100:.1f}",
                     color=color)
        plt.axhline(y=LN10, color="black", linestyle="--", linewidth=1)
        plt.xlabel("Batch Number")
        plt.ylabel("Average CE Train")
        plt.title(f"Conv-MNIST — CE Train\nPrune: {prune_layer}, BS={bs}")
        plt.grid(True)
        plt.tight_layout()
        out_png = os.path.join(plot_dir, f"CE_Train_Avg_{prune_layer}_BS{bs}.png")
        plt.savefig(out_png, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"[PLOT] {out_png}")

print("Done.")
